In [4]:
import pandas as pd
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import joblib
import re
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
import pickle

# Load Labeled Dataset with proper label handling
# Load Labeled Dataset with proper label handling
labeled_df = pd.read_csv("bert_predictions_test(VADER).csv")
labeled_df['Final_Comment'] = labeled_df['Final_Comment'].astype(str)

# Create label mapping and convert sentiment to numeric
sentiment_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
reverse_mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

# First, clean the sentiment column
if labeled_df['Sentiment'].dtype == object:
    # Remove any rows with NaN or empty strings in sentiment column
    labeled_df = labeled_df.dropna(subset=['Sentiment'])
    labeled_df = labeled_df[labeled_df['Sentiment'].astype(str).str.strip() != '']

    # Convert string labels to numeric
    labeled_df['Sentiment'] = labeled_df['Sentiment'].str.strip().map(sentiment_mapping)

# Now safely convert to integer
try:
    # First fill any remaining NaN with neutral (1) as a default
    labeled_df['Sentiment'] = labeled_df['Sentiment'].fillna(1)
    # Then convert to integer
    labeled_df['Sentiment'] = labeled_df['Sentiment'].astype(int)
    # Ensure all values are 0, 1, or 2
    labeled_df = labeled_df[labeled_df['Sentiment'].between(0, 2, inclusive='both')]
except (ValueError, TypeError) as e:
    print("Error converting sentiment labels to numeric values")
    print("Unique values found:", labeled_df['Sentiment'].unique())
    raise e

# Verify we have the expected values
print("Sentiment value counts:")
print(labeled_df['Sentiment'].value_counts())


# Preprocessing Function
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()


# Load Models
# Load TextBlob model
# Define the missing function first
def classify_sentiment(text):
    analysis = TextBlob(text)
    if analysis.sentiment.polarity > 0:
        return 2  # Positive
    elif analysis.sentiment.polarity == 0:
        return 1  # Neutral
    else:
        return 0  # Negative

# Then proceed with loading the models
try:
    with open('textblob_sentiment.pkl', 'rb') as f:
        textblob_model = pickle.load(f)
except AttributeError as e:
    print("Error loading TextBlob model. Using direct TextBlob analysis instead.")
    def textblob_analysis(texts):
        predictions = []
        for text in texts:
            analysis = TextBlob(text)
            if analysis.sentiment.polarity > 0:
                predictions.append(2)
            elif analysis.sentiment.polarity == 0:
                predictions.append(1)
            else:
                predictions.append(0)
        return np.array(predictions)

try:
    with open('vader_sentiment_model.pkl', 'rb') as f:
        vader_model = pickle.load(f)
except AttributeError as e:
    print("Error loading VADER model. Using direct VADER analysis instead.")
    sia = SentimentIntensityAnalyzer()
    def vader_analysis(texts):
        predictions = []
        for text in texts:
            scores = sia.polarity_scores(text)
            if scores['compound'] >= 0.05:
                predictions.append(2)
            elif scores['compound'] <= -0.05:
                predictions.append(0)
            else:
                predictions.append(1)
        return np.array(predictions)

# Load BERT models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load BERT TextBlob model
bert_tokenizer_textblob = BertTokenizer.from_pretrained('./bert_finetuned_textblob_model')
bert_model_textblob = BertForSequenceClassification.from_pretrained('./bert_finetuned_textblob_model')
bert_model_textblob.to(device)

# Load BERT VADER model
bert_tokenizer_vader = BertTokenizer.from_pretrained('./bert_finetuned_vader_model')
bert_model_vader = BertForSequenceClassification.from_pretrained('./bert_finetuned_vader_model')
bert_model_vader.to(device)

# Load Naive Bayes models
complementnb_model_textblob = joblib.load('complementnb_model_textblob.joblib')
complementnb_model_vader = joblib.load('complementnb_model_vader.joblib')


# Perform Sentiment Analysis Using TextBlob + Naive Bayes
def textblob_analysis(texts):
    return complementnb_model_textblob.predict(texts)


# Perform Sentiment Analysis Using VADER + Naive Bayes
def vader_analysis(texts):
    return complementnb_model_vader.predict(texts)


# Perform Sentiment Analysis Using BERT TextBlob model
def bert_textblob_analysis(texts):
    texts = texts.tolist()
    inputs = bert_tokenizer_textblob(texts, padding=True, truncation=True, return_tensors="pt", max_length=128)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        outputs = bert_model_textblob(**inputs)
    predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()
    return predictions


# Perform Sentiment Analysis Using BERT VADER model
def bert_vader_analysis(texts):
    texts = texts.tolist()
    inputs = bert_tokenizer_vader(texts, padding=True, truncation=True, return_tensors="pt", max_length=128)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        outputs = bert_model_vader(**inputs)
    predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()
    return predictions


# Split Data into Features and Labels
X = labeled_df['Final_Comment']
y = labeled_df['Sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply Sentiment Analysis for Each Model
test_predictions = pd.DataFrame({
    'Final_Comment': X_test,
    'Sentiment': y_test,
    'TextBlob_Prediction': textblob_analysis(X_test),
    'VADER_Prediction': vader_analysis(X_test),
    'BERT_TextBlob_Prediction': bert_textblob_analysis(X_test),
    'BERT_VADER_Prediction': bert_vader_analysis(X_test)
})

# Add string labels for visualization
test_predictions['Sentiment_Label'] = test_predictions['Sentiment'].map(reverse_mapping)
test_predictions['TextBlob_Label'] = test_predictions['TextBlob_Prediction'].map(reverse_mapping)
test_predictions['VADER_Label'] = test_predictions['VADER_Prediction'].map(reverse_mapping)
test_predictions['BERT_TextBlob_Label'] = test_predictions['BERT_TextBlob_Prediction'].map(reverse_mapping)
test_predictions['BERT_VADER_Label'] = test_predictions['BERT_VADER_Prediction'].map(reverse_mapping)


# Cross-Validation for Robust Evaluation
def run_cross_validation(pipeline_model, X, y):
    """Run 5-fold cross-validation on a full pipeline model"""
    scores = cross_val_score(pipeline_model, X, y, cv=5, scoring='accuracy')
    print(f"Cross-Validation Accuracy: {np.mean(scores):.2f} ± {np.std(scores):.2f}")


print("TextBlob Model Cross-Validation:")
run_cross_validation(complementnb_model_textblob, X, y)

print("VADER Model Cross-Validation:")
run_cross_validation(complementnb_model_vader, X, y)


# Hyperparameter Tuning for Naive Bayes Models
def tune_naive_bayes(pipeline_model, X, y):
    """Use GridSearchCV to tune ComplementNB alpha inside pipeline"""
    parameters = {'complementnb__alpha': [0.1, 0.5, 1.0, 2.0]}
    clf = GridSearchCV(pipeline_model, parameters, cv=3)
    clf.fit(X, y)
    print(f"Best parameters: {clf.best_params_}")
    return clf.best_estimator_


print("Tuning TextBlob Model:")
best_textblob_model = tune_naive_bayes(complementnb_model_textblob, X_train, y_train)

print("Tuning VADER Model:")
best_vader_model = tune_naive_bayes(complementnb_model_vader, X_train, y_train)


# Evaluate Models with Proper Methodology and Additional Metrics
def evaluate_model(y_true, y_pred, model_name):
    print(f"{model_name} Classification Report:")
    print(classification_report(y_true, y_pred))

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted'
    )
    print(f"Weighted Precision: {precision:.3f}")
    print(f"Weighted Recall: {recall:.3f}")
    print(f"Weighted F1-Score: {f1:.3f}")

    return accuracy_score(y_true, y_pred)


# Evaluate each model
textblob_acc = evaluate_model(test_predictions['Sentiment'],
                              test_predictions['TextBlob_Prediction'],
                              "TextBlob")

vader_acc = evaluate_model(test_predictions['Sentiment'],
                           test_predictions['VADER_Prediction'],
                           "VADER")

bert_textblob_acc = evaluate_model(test_predictions['Sentiment'],
                                   test_predictions['BERT_TextBlob_Prediction'],
                                   "BERT TextBlob")

bert_vader_acc = evaluate_model(test_predictions['Sentiment'],
                                test_predictions['BERT_VADER_Prediction'],
                                "BERT VADER")


# Confusion Matrices for Each Model (using string labels)
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Neutral', 'Positive'],
                yticklabels=['Negative', 'Neutral', 'Positive'])
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


plot_confusion_matrix(test_predictions['Sentiment'],
                      test_predictions['TextBlob_Prediction'],
                      "TextBlob Confusion Matrix")

plot_confusion_matrix(test_predictions['Sentiment'],
                      test_predictions['VADER_Prediction'],
                      "VADER Confusion Matrix")

plot_confusion_matrix(test_predictions['Sentiment'],
                      test_predictions['BERT_TextBlob_Prediction'],
                      "BERT TextBlob Confusion Matrix")

plot_confusion_matrix(test_predictions['Sentiment'],
                      test_predictions['BERT_VADER_Prediction'],
                      "BERT VADER Confusion Matrix")


# Sentiment Distribution (using string labels)
def plot_sentiment_distribution(df, title):
    plt.figure(figsize=(8, 5))
    sns.countplot(x='TextBlob_Label', data=df, palette='coolwarm',
                  order=['Negative', 'Neutral', 'Positive'])
    plt.title(title + " (TextBlob)")
    plt.xlabel("Sentiment")
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.countplot(x='VADER_Label', data=df, palette='coolwarm',
                  order=['Negative', 'Neutral', 'Positive'])
    plt.title(title + " (VADER)")
    plt.xlabel("Sentiment")
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.countplot(x='BERT_TextBlob_Label', data=df, palette='coolwarm',
                  order=['Negative', 'Neutral', 'Positive'])
    plt.title(title + " (BERT TextBlob)")
    plt.xlabel("Sentiment")
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.countplot(x='BERT_VADER_Label', data=df, palette='coolwarm',
                  order=['Negative', 'Neutral', 'Positive'])
    plt.title(title + " (BERT VADER)")
    plt.xlabel("Sentiment")
    plt.ylabel("Count")
    plt.show()


plot_sentiment_distribution(test_predictions, "Sentiment Distribution in Test Data")


# Word Clouds for Sentiments
def generate_wordcloud(words, title):
    if not words:
        print(f"Warning: No words available for {title}")
        return
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(words))
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(title)
    plt.show()


positive_words = ' '.join(
    test_predictions[test_predictions['TextBlob_Prediction'] == 2]['Final_Comment'].apply(preprocess_text)).split()
neutral_words = ' '.join(
    test_predictions[test_predictions['TextBlob_Prediction'] == 1]['Final_Comment'].apply(preprocess_text)).split()
negative_words = ' '.join(
    test_predictions[test_predictions['TextBlob_Prediction'] == 0]['Final_Comment'].apply(preprocess_text)).split()

generate_wordcloud(positive_words, "Word Cloud for Positive Sentiment")
generate_wordcloud(neutral_words, "Word Cloud for Neutral Sentiment")
generate_wordcloud(negative_words, "Word Cloud for Negative Sentiment")


# Compare Models' Performance
def compare_models_performance(accuracies):
    models = ['TextBlob', 'VADER', 'BERT TextBlob', 'BERT VADER']
    plt.figure(figsize=(10, 6))
    sns.barplot(x=models, y=accuracies, palette='coolwarm')
    plt.title("Comparison of Models' Performance")
    plt.xlabel('Model')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0)
    plt.show()


compare_models_performance([textblob_acc, vader_acc, bert_textblob_acc, bert_vader_acc])

Sentiment value counts:
Sentiment
1    1787
Name: count, dtype: int64
Error loading VADER model. Using direct VADER analysis instead.
TextBlob Model Cross-Validation:
Cross-Validation Accuracy: 1.00 ± 0.00
VADER Model Cross-Validation:
Cross-Validation Accuracy: 1.00 ± 0.00
Tuning TextBlob Model:


ValueError: Invalid parameter 'complementnb' for estimator Pipeline(steps=[('tfidf', TfidfVectorizer(max_df=0.8)),
                ('nb', ComplementNB(alpha=0.5))]). Valid parameters are: ['memory', 'steps', 'transform_input', 'verbose'].